# Phase-2 Step 3: Baseline Models

**Objective:** Train and evaluate baseline models for SOH prediction.

**Models:**
1. LSTM
2. CNN-LSTM
3. Transformer (small)

**Methodology:**
- 3 runs per model (reproducibility)
- Fixed random seeds
- Early stopping
- Identical train/test splits

**Metrics:**
- MAE (Mean Absolute Error)
- RMSE (Root Mean Square Error)
- MAPE (Mean Absolute Percentage Error)
- R² (Coefficient of Determination)

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os

# ============================================================================
# SETUP
# ============================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import torch

from baseline_models import (
    BaselineTrainer, 
    set_seed,
    LSTMModel,
    CNNLSTMModel,
    TransformerModel
)
from utils import plot_style_setup

plot_style_setup()

# Load config
with open('phase2_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Set master seed
set_seed(config['random_seed'])

print("✓ Imports complete")
print(f"✓ Configuration loaded")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Load Harmonized Data

In [ ]:
# Find the latest harmonized data
harmonized_dir = os.path.join(PROJECT_ROOT, 'data/phase2/harmonized')

# Load train and test features
train_path = os.path.join(harmonized_dir, 'train_features.csv')
test_path = os.path.join(harmonized_dir, 'test_features.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"\nFeatures: {len(train_df.columns)} columns")

# Show columns
print("\nColumns:")
for col in train_df.columns:
    print(f"  - {col}")

In [ ]:
# Define feature columns (exclude identifiers and target)
exclude_cols = ['battery_id', 'cycle_number', 'original_index', 'soh', 'has_spikes']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

print(f"\nUsing {len(feature_cols)} features for training:")
for col in feature_cols:
    print(f"  - {col}")

In [ ]:
# Quick look at the data
print("\nTrain set statistics:")
train_df[feature_cols + ['soh']].describe()

## 2. Initialize Trainer

In [ ]:
# Create trainer
trainer = BaselineTrainer(config, PROJECT_ROOT)

print(f"\nTraining configuration:")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Early stopping patience: {config['training']['early_stopping']['patience']}")
print(f"  Runs per model: {config['training']['num_runs']}")

## 3. Train All Baseline Models

In [ ]:
# Run all experiments
# This trains LSTM, CNN-LSTM, and Transformer, each 3 times

results_df = trainer.run_experiments(
    train_df, 
    test_df, 
    feature_cols,
    models=['lstm', 'cnn_lstm', 'transformer'],
    num_runs=config['training']['num_runs']
)

## 4. Analyze Results

In [ ]:
# Display results DataFrame
print("\nAll experiment results:")
results_df

In [ ]:
# Summary statistics
summary = results_df.groupby('model').agg({
    'mae': ['mean', 'std'],
    'rmse': ['mean', 'std'],
    'r2': ['mean', 'std'],
    'training_time': 'mean'
}).round(4)

print("\nSummary Statistics (mean ± std):")
summary

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['mae', 'rmse', 'r2']
titles = ['MAE (lower is better)', 'RMSE (lower is better)', 'R² (higher is better)']
colors = ['#2ecc71', '#3498db', '#e74c3c']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    # Box plot
    results_df.boxplot(column=metric, by='model', ax=ax)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Model')
    ax.set_ylabel(metric.upper())
    
    # Remove automatic title
    plt.suptitle('')

plt.tight_layout()
plt.savefig('results/phase2/figures/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/baseline_comparison.png")

In [ ]:
# Bar plot with error bars
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, metrics):
    means = results_df.groupby('model')[metric].mean()
    stds = results_df.groupby('model')[metric].std()
    
    x = np.arange(len(means))
    bars = ax.bar(x, means, yerr=stds, capsize=5, 
                  color=['#3498db', '#2ecc71', '#e74c3c'],
                  edgecolor='black', linewidth=1.2)
    
    ax.set_xticks(x)
    ax.set_xticklabels(means.index, fontsize=11)
    ax.set_ylabel(metric.upper(), fontsize=12)
    ax.set_title(f'{metric.upper()} by Model', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, mean, std in zip(bars, means, stds):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + std + 0.01,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('results/phase2/figures/baseline_bar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/baseline_bar_comparison.png")

## 5. Training Curves Analysis

In [ ]:
# Plot training curves from the best run of each model
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

model_names = ['lstm', 'cnn_lstm', 'transformer']
colors = {'train': '#3498db', 'val': '#e74c3c'}

for ax, model_name in zip(axes, model_names):
    # Get the best run for this model
    model_results = [r for r in trainer.results if r.model_name == model_name]
    best_result = min(model_results, key=lambda x: x.test_rmse)
    
    epochs = range(1, len(best_result.train_loss) + 1)
    
    ax.plot(epochs, best_result.train_loss, label='Train', color=colors['train'])
    ax.plot(epochs, best_result.val_loss, label='Validation', color=colors['val'])
    ax.axvline(x=best_result.best_epoch, color='green', linestyle='--', 
               label=f'Best epoch: {best_result.best_epoch}')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.set_title(f'{model_name.upper()} - Training Curves', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/phase2/figures/baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/baseline_training_curves.png")

## 6. Prediction Analysis

In [ ]:
# Get predictions from best models
from torch.utils.data import DataLoader
from baseline_models import SOHDataset

# Prepare test data
X_test = test_df[feature_cols].values
y_test = test_df['soh'].values

test_dataset = SOHDataset(X_test, y_test, sequence_length=10)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Get predictions for each model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
predictions = {}

for model_name, model in trainer.best_models.items():
    model.eval()
    preds = []
    targets = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            preds.extend(outputs.cpu().numpy())
            targets.extend(batch_y.numpy())
    
    predictions[model_name] = {
        'preds': np.array(preds),
        'targets': np.array(targets)
    }

print(f"✓ Got predictions for {len(predictions)} models")

In [ ]:
# Prediction vs Actual plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (model_name, pred_data) in zip(axes, predictions.items()):
    preds = pred_data['preds']
    targets = pred_data['targets']
    
    # Scatter plot
    ax.scatter(targets, preds, alpha=0.5, s=30)
    
    # Perfect prediction line
    min_val = min(targets.min(), preds.min())
    max_val = max(targets.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
    
    # Compute R²
    ss_res = np.sum((targets - preds) ** 2)
    ss_tot = np.sum((targets - np.mean(targets)) ** 2)
    r2 = 1 - ss_res / ss_tot
    
    ax.set_xlabel('Actual SOH (%)', fontsize=11)
    ax.set_ylabel('Predicted SOH (%)', fontsize=11)
    ax.set_title(f'{model_name.upper()}\nR² = {r2:.4f}', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/phase2/figures/baseline_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/baseline_predictions.png")

In [ ]:
# Error distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (model_name, pred_data) in zip(axes, predictions.items()):
    errors = pred_data['preds'] - pred_data['targets']
    
    ax.hist(errors, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='r', linestyle='--', label='Zero error')
    ax.axvline(x=np.mean(errors), color='g', linestyle='--', 
               label=f'Mean: {np.mean(errors):.3f}')
    
    ax.set_xlabel('Prediction Error (%)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{model_name.upper()} - Error Distribution', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/phase2/figures/baseline_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/baseline_error_distribution.png")

## 7. Save Results

In [ ]:
# Save all results
trainer.save_results(results_df)

In [ ]:
# Print final summary
trainer.print_summary(results_df)

## 8. Model Comparison Table

In [ ]:
# Create a nice comparison table
comparison = []

for model_name in ['lstm', 'cnn_lstm', 'transformer']:
    model_runs = results_df[results_df['model'] == model_name]
    
    comparison.append({
        'Model': model_name.upper(),
        'MAE (mean ± std)': f"{model_runs['mae'].mean():.4f} ± {model_runs['mae'].std():.4f}",
        'RMSE (mean ± std)': f"{model_runs['rmse'].mean():.4f} ± {model_runs['rmse'].std():.4f}",
        'R² (mean ± std)': f"{model_runs['r2'].mean():.4f} ± {model_runs['r2'].std():.4f}",
        'Avg Training Time (s)': f"{model_runs['training_time'].mean():.1f}"
    })

comparison_df = pd.DataFrame(comparison)
print("\n" + "="*80)
print("BASELINE MODEL COMPARISON TABLE")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Save comparison table
comparison_df.to_csv('results/phase2/baseline_comparison_table.csv', index=False)
print("\n✓ Saved: results/phase2/baseline_comparison_table.csv")

## Summary

**Step 3 Completed:**

✓ LSTM model trained (3 runs)  
✓ CNN-LSTM model trained (3 runs)  
✓ Transformer model trained (3 runs)  
✓ Results saved with mean ± std  
✓ Best models saved for later use  
✓ Visualization plots generated  

**Output Files:**
- `models/phase2/baselines/lstm_best_*.pt`
- `models/phase2/baselines/cnn_lstm_best_*.pt`
- `models/phase2/baselines/transformer_best_*.pt`
- `results/phase2/baseline_results_*.csv`
- `results/phase2/baseline_summary_*.csv`
- `results/phase2/baseline_comparison_table.csv`
- `results/phase2/figures/baseline_*.png`

**Next Step:** Step 4 - Multimodal Model (Physics + Semantic)